# Semana 15 — MLOps básico con Weights & Biases

**Especialización en Deep Learning — Universidad de Cundinamarca**

**Equipo:**
- Laura Amado — Investigación teórica
- Harold Duque — ML Engineer (este notebook)
- Miguel Ángel Córdoba — Análisis de experimentos
- Jensul Villalba — Documentación y presentación

## Objetivo
Implementar un flujo end-to-end de Deep Learning sobre MNIST que evidencie los 4 pilares de MLOps:
1. **Data Journey** — origen y recorrido del dato
2. **Acceso y manipulación de datos** — pipelines reproducibles
3. **Monitoreo y logging** — todo loggeado en W&B
4. **Model Serving** — guardado del modelo como artifact (concepto)

## Contenido
1. Setup de W&B
2. Carga y manipulación de datos
3. Definición del modelo
4. Entrenamiento con logging
5. Evaluación y predicciones
6. Guardado del modelo como artifact
7. (Opcional) Sweep de hiperparámetros

## 1. Setup de Weights & Biases

**Antes de correr esto:**
1. Vayan a https://wandb.ai/site → Sign Up (gratis)
2. Una vez registrados, vayan a https://wandb.ai/authorize y copien su API key
3. Al correr la siguiente celda les va a pedir esa key — péguenla

In [ ]:
!pip install -q wandb

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Sequential

import wandb
from wandb.integration.keras import WandbMetricsLogger

print('TensorFlow:', tf.__version__)
print('W&B:', wandb.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

# Semillas para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# Login a W&B — al correr esto les pedirá la API key
wandb.login()

## 2. Carga y manipulación de datos

**Data Journey aplicado a MNIST:**
- Fuente: `tf.keras.datasets.mnist` (dataset público)
- Ingesta: `load_data()` → 60K train + 10K test
- Manipulación: normalización a `[0, 1]` + agregar canal
- Particionamiento: 80/20 dentro del train + test set separado
- Carga eficiente: `tf.data.Dataset` con `shuffle` + `batch` + `prefetch`

In [ ]:
(x_full, y_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Manipulación: normalizar y agregar canal
x_full = x_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_full = np.expand_dims(x_full, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# One-hot encoding para las etiquetas
y_full = tf.keras.utils.to_categorical(y_full, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# Partición 80/20 dentro del set de 60K
n_train = int(0.8 * len(x_full))
x_train, x_val = x_full[:n_train], x_full[n_train:]
y_train, y_val = y_full[:n_train], y_full[n_train:]

print(f'Train: {x_train.shape}  |  Val: {x_val.shape}  |  Test: {x_test.shape}')
print(f'Rango: [{x_train.min()}, {x_train.max()}]')

In [ ]:
# Visualización del dato — evidencia del Data Journey
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.set_title(f'Label: {np.argmax(y_train[i])}', fontsize=10)
    ax.axis('off')
plt.suptitle('Muestras del dataset MNIST')
plt.tight_layout()
plt.show()

## 3. Definición del modelo (CNN simple)

Arquitectura:
- 2 bloques `Conv2D + MaxPool2D` para extraer features visuales
- Capa densa intermedia con `Dropout` para regularizar
- Capa de salida con 10 neuronas (softmax) para los 10 dígitos

In [ ]:
def build_cnn(num_filters=32, dense_units=128, dropout=0.3):
    model = Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(num_filters, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Conv2D(num_filters * 2, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dropout(dropout),
        layers.Dense(dense_units, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ], name='mnist_cnn')
    return model

model = build_cnn()
model.summary()

## 4. Entrenamiento con logging en W&B

**Lo importante:** todo lo que ocurre durante el entrenamiento queda registrado en W&B y puede verse en el dashboard en tiempo real.

El callback `WandbMetricsLogger` loggea automáticamente:
- `train_loss`, `train_accuracy`
- `val_loss`, `val_accuracy`
- `learning_rate`

In [ ]:
# === Configuración del run — CAMBIEN ESTOS VALORES PARA HACER MÚLTIPLES RUNS ===
config = {
    'epochs': 8,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'optimizer': 'adam',
    'num_filters': 32,
    'dense_units': 128,
    'dropout': 0.3,
    'architecture': 'CNN-2bloques',
    'dataset': 'MNIST',
}

# Inicializar el run en W&B
run = wandb.init(
    project='unicundi-deeplearning-w15',
    name=f'cnn-{config["optimizer"]}-lr{config["learning_rate"]}-bs{config["batch_size"]}',
    config=config,
    notes='Run base con hiperparámetros por defecto',
    tags=['mnist', 'cnn', 'experimento-base'],
)
cfg = wandb.config

In [ ]:
# Construir el modelo con la configuración del run
model = build_cnn(
    num_filters=cfg.num_filters,
    dense_units=cfg.dense_units,
    dropout=cfg.dropout,
)

# Seleccionar el optimizador
if cfg.optimizer == 'adam':
    optimizer = tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate)
elif cfg.optimizer == 'sgd':
    optimizer = tf.keras.optimizers.SGD(learning_rate=cfg.learning_rate, momentum=0.9)
else:
    optimizer = tf.keras.optimizers.RMSprop(learning_rate=cfg.learning_rate)

model.compile(optimizer=optimizer,
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Datasets de TF
train_ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
            .shuffle(10_000, seed=42)
            .batch(cfg.batch_size)
            .prefetch(tf.data.AUTOTUNE))
val_ds = (tf.data.Dataset.from_tensor_slices((x_val, y_val))
          .batch(cfg.batch_size)
          .prefetch(tf.data.AUTOTUNE))

# Entrenamiento — W&B loggea automáticamente las métricas
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=cfg.epochs,
    callbacks=[WandbMetricsLogger(log_freq='epoch')],
    verbose=1,
)

## 5. Evaluación final y tabla de predicciones

Evaluamos sobre el **test set** (datos que el modelo nunca vio) y loggeamos:
- Métrica final: `test_loss`, `test_accuracy`
- Tabla de predicciones de muestra con imágenes (la verán en W&B como una tabla interactiva)

In [ ]:
# Evaluación sobre el test set
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(cfg.batch_size)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Test loss: {test_loss:.4f}  |  Test accuracy: {test_acc:.4f}')

wandb.log({'test_loss': test_loss, 'test_accuracy': test_acc})

In [ ]:
# Tabla de predicciones de muestra en W&B
sample_preds = model.predict(x_test[:20], verbose=0)
sample_labels = np.argmax(sample_preds, axis=1)
sample_true = np.argmax(y_test[:20], axis=1)

table = wandb.Table(columns=['imagen', 'real', 'predicción', 'correcto'])
for i in range(20):
    table.add_data(
        wandb.Image(x_test[i]),
        int(sample_true[i]),
        int(sample_labels[i]),
        bool(sample_true[i] == sample_labels[i]),
    )
wandb.log({'predicciones_de_muestra': table})
print('Tabla de predicciones subida a W&B')

In [ ]:
# Matriz de confusión
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_pred_all = model.predict(x_test, verbose=0)
y_pred_class = np.argmax(y_pred_all, axis=1)
y_true_class = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true_class, y_pred_class)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de confusión — Test set')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

wandb.log({'confusion_matrix': wandb.Image('confusion_matrix.png')})

## 6. Guardado del modelo como Artifact (concepto de Model Serving)

Aquí se aplica el concepto de Model Serving: guardamos el modelo entrenado en un formato (`.keras`) listo para ser cargado por un servidor de inferencia. W&B versiona el artifact automáticamente.

In [ ]:
os.makedirs('models', exist_ok=True)
model_path = 'models/mnist_classifier.keras'
model.save(model_path)

artifact = wandb.Artifact(
    name='mnist_classifier',
    type='model',
    description=f'CNN entrenada con {cfg.optimizer} lr={cfg.learning_rate} bs={cfg.batch_size}',
    metadata=dict(cfg),
)
artifact.add_file(model_path)
run.log_artifact(artifact)

print(f'Modelo guardado y subido como artifact a W&B')

In [ ]:
# Cerrar el run
wandb.finish()

## 7. (Opcional) Hacer múltiples runs cambiando hiperparámetros

**Para tener material de comparación en el dashboard**, repitan las celdas de "Configuración del run" → "Entrenamiento" → "Cerrar el run" cambiando algún hiperparámetro cada vez.

**Recomendado: hacer al menos 3 runs distintos.** Sugerencias de variaciones:

| Run | learning_rate | batch_size | optimizer |
|---|---|---|---|
| 1 (base) | 0.001 | 64 | adam |
| 2 | 0.0001 | 64 | adam |
| 3 | 0.001 | 128 | sgd |
| 4 (extra) | 0.005 | 32 | rmsprop |

Después de los 3-4 runs, vayan al dashboard de W&B y comparen los runs.

## 8. ¿Qué pasó y dónde verlo?

1. Vayan a https://wandb.ai/<su-usuario>/unicundi-deeplearning-w15
2. Verán todos los runs en una tabla
3. Pueden seleccionar varios y comparar curvas
4. La tabla `predicciones_de_muestra` se ve interactiva en el panel del run
5. El artifact `mnist_classifier` queda en la pestaña 'Artifacts'

**Para hacer público el proyecto** (necesario para que el profe lo vea):
- Project settings → Privacy → 'Open' o 'Public'